In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_csv("LoanApprovalPrediction.csv")

In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            598 non-null    str    
 1   Gender             598 non-null    str    
 2   Married            598 non-null    str    
 3   Dependents         586 non-null    float64
 4   Education          598 non-null    str    
 5   Self_Employed      598 non-null    str    
 6   ApplicantIncome    598 non-null    int64  
 7   CoapplicantIncome  598 non-null    float64
 8   LoanAmount         577 non-null    float64
 9   Loan_Amount_Term   584 non-null    float64
 10  Credit_History     549 non-null    float64
 11  Property_Area      598 non-null    str    
 12  Loan_Status        598 non-null    str    
dtypes: float64(5), int64(1), str(7)
memory usage: 60.9 KB


In [4]:
# Handling Missing Value
data.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0.0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1.0,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0.0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0.0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0.0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [5]:
data.isnull().sum()

Loan_ID               0
Gender                0
Married               0
Dependents           12
Education             0
Self_Employed         0
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           21
Loan_Amount_Term     14
Credit_History       49
Property_Area         0
Loan_Status           0
dtype: int64

In [6]:
data.drop(['Loan_ID'],axis=1,inplace=True)

In [7]:
data['Dependents'] = data['Dependents'].fillna(data['Dependents'].mode()[0])

In [8]:
data.Loan_Amount_Term.unique()

array([360., 120., 240.,  nan, 180.,  60., 300., 480.,  36.,  84.,  12.])

In [9]:
data['Loan_Amount_Term'] = data['Loan_Amount_Term'].fillna(data['Loan_Amount_Term'].median())

In [10]:
data['LoanAmount'] = data['LoanAmount'].fillna(data['LoanAmount'].median())

In [11]:
data['Credit_History'] = data['Credit_History'].fillna(data['Credit_History'].mode()[0])

In [12]:
encoding = {
    'Gender': {'Male':1 , 'Female': 0}, 
    'Married': {'Yes': 1, 'No': 0},
    'Dependents': {'0':0, '1':1, '2': 2, '4': 4},
    'Education': {'Graduate': 1, 'Not Graduate': 0},
    'Self_Employed': {'Yes': 1, 'No': 0},
    'Property_Area': {'Rural': 0, 'Semiurban': 2, 'Urban': 1},
    'Loan_Status': {'Y': 1, 'N': 0}
}

In [13]:
data = data.replace(encoding, inplace=True)

In [16]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             598 non-null    int64  
 1   Married            598 non-null    int64  
 2   Dependents         598 non-null    float64
 3   Education          598 non-null    int64  
 4   Self_Employed      598 non-null    int64  
 5   ApplicantIncome    598 non-null    int64  
 6   CoapplicantIncome  598 non-null    float64
 7   LoanAmount         598 non-null    float64
 8   Loan_Amount_Term   598 non-null    float64
 9   Credit_History     598 non-null    float64
 10  Property_Area      598 non-null    int64  
 11  Loan_Status        598 non-null    int64  
dtypes: float64(5), int64(7)
memory usage: 56.2 KB


In [20]:
from sklearn.model_selection import train_test_split

X = data.drop('Loan_Status', axis = 1)
y = data['Loan_Status']

In [21]:
from sklearn.preprocessing import StandardScaler
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

In [22]:
from sklearn import preprocessing
  
# label_encoder object knows how 
# to understand word labels.
label_encoder = preprocessing.LabelEncoder()
obj = (data.dtypes == 'object')
for col in list(obj[obj].index):
  data[col] = label_encoder.fit_transform(data[col])

In [23]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV

def evaluate_model(model):
    X_train, X_test, y_train, y_test  = train_test_split(X, y, test_size = 0.2, random_state = 42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    cross_val = cross_val_score(model, X, y, cv=5)
    avg_cross_val = np.mean(cross_val)
    print(f"{model.__class__.__name__} - Accuarcy : {accuracy: .2f} , Cross-Val-Score: {avg_cross_val: .2f}")
    return avg_cross_val

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    LogisticRegression(),
    svm.SVC(),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    GradientBoostingClassifier(),
    KNeighborsClassifier(n_neighbors=3)
}

In [27]:
model_score = {model.__class__.__name__:evaluate_model(model) for model in models}

KNeighborsClassifier - Accuarcy :  0.70 , Cross-Val-Score:  0.74
RandomForestClassifier - Accuarcy :  0.81 , Cross-Val-Score:  0.80
DecisionTreeClassifier - Accuarcy :  0.68 , Cross-Val-Score:  0.71
GradientBoostingClassifier - Accuarcy :  0.79 , Cross-Val-Score:  0.78
LogisticRegression - Accuarcy :  0.82 , Cross-Val-Score:  0.80
SVC - Accuarcy :  0.82 , Cross-Val-Score:  0.80


In [28]:
def tune_model(model, param_grid):
    tuner = RandomizedSearchCV(model, param_grid, cv = 5, n_iter =20, verbose = True, random_state = 42)
    tuner.fit(X, y)
    print(f"Best Score for {model.__class__.__name__}: {tuner.best_score_:.2f}")
    print(f"Best Parameter for {model.__class__.__name__}: {tuner.best_params_}")
    return tuner.best_estimator_

In [42]:
log_reg_grid = {'C': np.logspace(-4, 4, 20), "solver": ["liblinear"]}

svc_grid = {'C': [0.25, 0.50, 0.75, 1], "kernel": ['linear']}

dt__grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = {
    'n_estimators': np.arange(10, 1000, 10),
    'max_features': ['log2', 'sqrt'], 
    'max_depth': [None, 3, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 20, 50, 100],
    'min_samples_leaf': [1, 2, 5, 10]
}

gb_grid = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0]
}

kn_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

In [43]:
best_log_reg = tune_model(LogisticRegression(), log_reg_grid)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Score for LogisticRegression: 0.81
Best Parameter for LogisticRegression: {'solver': 'liblinear', 'C': np.float64(0.23357214690901212)}


In [31]:
best_svc_reg = tune_model(svm.SVC(), svc_grid)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Score for SVC: 0.81
Best Parameter for SVC: {'kernel': 'linear', 'C': 0.25}


/Users/carlos/Analisis de Datos/proyectos/loan-approval-prediction/.venv/lib/python3.14/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [32]:
best_dt = tune_model(DecisionTreeClassifier(), dt__grid)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Score for DecisionTreeClassifier: 0.81
Best Parameter for DecisionTreeClassifier: {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 3, 'criterion': 'entropy'}


In [33]:
best_rf = tune_model(RandomForestClassifier(), rf_grid)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Score for RandomForestClassifier: 0.81
Best Parameter for RandomForestClassifier: {'n_estimators': np.int64(930), 'min_samples_split': 50, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 30}


In [36]:
best_gb = tune_model(GradientBoostingClassifier(), gb_grid)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Score for GradientBoostingClassifier: 0.80
Best Parameter for GradientBoostingClassifier: {'subsample': 1.0, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.01}


In [38]:
best_kn = tune_model(KNeighborsClassifier(), kn_grid)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Score for KNeighborsClassifier: 0.75
Best Parameter for KNeighborsClassifier: {'weights': 'distance', 'n_neighbors': 9, 'metric': 'manhattan'}


In [44]:
import joblib
final_model = best_rf
joblib.dump(final_model, 'loan_status_predictor_rf.pkl')

['loan_status_predictor_rf.pkl']

In [51]:
# Prediction System

sample_data = pd.DataFrame({
    'Gender': [1],
    'Married': [1],
    'Dependents': [2],
    'Education': [0],
    'Self_Employed': [0],
    'ApplicantIncome': [2000],
    'CoapplicantIncome': [0.0],
    'LoanAmount': [150],
    'Loan_Amount_Term': [180],
    'Credit_History': [1],
    'Property_Area': [1]
})

sample_data[num_cols] = scaler.transform(sample_data[num_cols])
loaded_model = joblib.load('loan_status_predictor_rf.pkl')
prediction = loaded_model.predict(sample_data)
print(prediction[0])

result = "Prestamo aprobado" if prediction[0] == 1 else "Prestamo no aprobado"
print(f"\nResultado de la prediccion: {result}")

1

Resultado de la prediccion: Prestamo aprobado


In [52]:
joblib.dump(scaler, 'vector.pkl')

['vector.pkl']